In [18]:
#                 USER QUERY
#                      │
#                      ▼
#              Shared Graph State
#                      │
#       ┌──────────────┼──────────────┐
#       ▼              ▼              ▼
#  Signal Agent   Billing Agent   Device Agent
#       │              │              │
#       └──────────────┼──────────────┘
#                      ▼
#              Aggregator Agent
#                      ▼
#                Final Report

In [21]:
# !pip install -q \
# langchain==0.3.25 \
# langgraph==0.4.7 \
# langchain-openai==0.3.18 \
# langchain-community==0.3.24 \
# langchain-core==0.3.60 \
# langchain-text-splitters==0.3.8 \
# pydantic==2.11.4 \
# faiss-cpu \
# tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.1 MB/s eta 0:00:00
ERROR: Cannot install langchain-core==0.3.60, langchain-openai==0.3.18, langchain==0.3.25 and langgraph==0.4.7 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [1]:
# ============================================================
# PARALLEL TELECOM MULTI-AGENT SYSTEM (UPDATED VERSION)
# ============================================================
#
# Features:
# - Latest LangChain Compatible
# - Latest LangGraph Compatible
# - Parallel Multi-Agent Execution
# - GPT-4o Mini
# - text-embedding-3-small
# - Telecom RAG
# - Google Colab Optimized
# - OpenAI Key from Colab Secrets
#
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q \
langchain \
langgraph \
langchain-openai \
langchain-community \
langchain-core \
faiss-cpu \
tiktoken

# ============================================================
# 2. IMPORTS
# ============================================================

from typing import TypedDict, Dict, Any

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# ============================================================
# 3. LOAD OPENAI API KEY FROM COLAB SECRETS
# ============================================================

# Google Colab:
# Left Sidebar → Secrets
#
# Add:
# Name  : OPENAI_API_KEY
# Value : your-openai-api-key

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ============================================================
# 4. INITIALIZE GPT-4o MINI
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# ============================================================
# 5. INITIALIZE EMBEDDING MODEL
# ============================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# ============================================================
# 6. TELECOM KNOWLEDGE BASE
# ============================================================

telecom_knowledge = [

    """
    Tower congestion can cause:
    - slow internet
    - packet loss
    - high latency
    - call drops
    during peak hours.
    """,

    """
    Billing throttling occurs after
    Fair Usage Policy (FUP) limits
    are exceeded by customers.
    """,

    """
    Incorrect APN configuration may affect:
    - LTE
    - VoLTE
    - mobile internet
    - connectivity
    """,

    """
    SIM card aging may result in:
    - weak signal
    - intermittent disconnects
    - authentication failures
    """,

    """
    Outdated firmware may create:
    - compatibility issues
    - unstable network behavior
    - call instability
    """,

    """
    Regional tower outages may impact:
    - multiple customers
    - enterprise clients
    - mobile internet
    - voice quality
    """
]

# ============================================================
# 7. CREATE DOCUMENT OBJECTS
# ============================================================

documents = [
    Document(page_content=doc)
    for doc in telecom_knowledge
]

# ============================================================
# 8. SPLIT DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)

# ============================================================
# 9. CREATE VECTOR DATABASE
# ============================================================

vectorstore = FAISS.from_documents(
    split_docs,
    embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# ============================================================
# 10. DEFINE SHARED GRAPH STATE
# ============================================================

class TelecomState(TypedDict):

    customer_query: str

    signal_result: Dict[str, Any]
    billing_result: Dict[str, Any]
    device_result: Dict[str, Any]

    final_report: str

# ============================================================
# 11. HELPER FUNCTION FOR RETRIEVAL
# ============================================================

def retrieve_context(query: str):

    docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])

    return context

# ============================================================
# 12. SIGNAL ANALYSIS AGENT
# ============================================================

def signal_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom Signal Analysis Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - signal quality
    - network congestion
    - tower outages
    - packet loss
    - call drops

    Return:
    1. Problem Summary
    2. Root Cause
    3. Severity
    """

    response = llm.invoke(prompt)

    return {
        "signal_result": {
            "analysis": response.content
        }
    }

# ============================================================
# 13. BILLING VALIDATION AGENT
# ============================================================

def billing_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom Billing Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - throttling
    - billing status
    - plan validity
    - FUP restrictions

    Return:
    1. Billing Status
    2. Possible Restrictions
    3. Recommended Action
    """

    response = llm.invoke(prompt)

    return {
        "billing_result": {
            "analysis": response.content
        }
    }

# ============================================================
# 14. DEVICE DIAGNOSTICS AGENT
# ============================================================

def device_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom Device Diagnostics Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - APN configuration
    - SIM issues
    - firmware issues
    - device compatibility

    Return:
    1. Device Health
    2. Possible Device Problems
    3. Recommended Fix
    """

    response = llm.invoke(prompt)

    return {
        "device_result": {
            "analysis": response.content
        }
    }

# ============================================================
# 15. AGGREGATOR AGENT
# ============================================================

def aggregator_agent(state: TelecomState):

    signal_analysis = state["signal_result"]["analysis"]
    billing_analysis = state["billing_result"]["analysis"]
    device_analysis = state["device_result"]["analysis"]

    prompt = f"""
    You are a Senior Telecom Operations Engineer.

    Combine all specialist reports into
    one professional telecom diagnosis.

    SIGNAL ANALYSIS:
    {signal_analysis}

    BILLING ANALYSIS:
    {billing_analysis}

    DEVICE ANALYSIS:
    {device_analysis}

    Generate:
    1. Executive Summary
    2. Root Cause
    3. Severity
    4. Recommended Actions
    5. Confidence Score

    Keep the response professional.
    """

    response = llm.invoke(prompt)

    return {
        "final_report": response.content
    }

# ============================================================
# 16. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(TelecomState)

# ============================================================
# 17. ADD NODES
# ============================================================

builder.add_node("signal_agent", signal_agent)

builder.add_node("billing_agent", billing_agent)

builder.add_node("device_agent", device_agent)

builder.add_node("aggregator_agent", aggregator_agent)

# ============================================================
# 18. DEFINE PARALLEL EXECUTION FLOW
# ============================================================

# START → PARALLEL AGENTS

builder.add_edge(START, "signal_agent")

builder.add_edge(START, "billing_agent")

builder.add_edge(START, "device_agent")

# PARALLEL AGENTS → AGGREGATOR

builder.add_edge("signal_agent", "aggregator_agent")

builder.add_edge("billing_agent", "aggregator_agent")

builder.add_edge("device_agent", "aggregator_agent")

# AGGREGATOR → END

builder.add_edge("aggregator_agent", END)

# ============================================================
# 19. COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# 20. SAMPLE CUSTOMER QUERY
# ============================================================

input_state = {

    "customer_query": """
    My mobile internet is extremely slow
    and calls are dropping frequently
    since yesterday evening.
    """
}

# ============================================================
# 21. RUN GRAPH
# ============================================================

result = graph.invoke(input_state)

# ============================================================
# 22. DISPLAY RESULTS
# ============================================================

print("\n" + "="*70)
print("SIGNAL ANALYSIS AGENT")
print("="*70)

print(result["signal_result"]["analysis"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("BILLING ANALYSIS AGENT")
print("="*70)

print(result["billing_result"]["analysis"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("DEVICE DIAGNOSTICS AGENT")
print("="*70)

print(result["device_result"]["analysis"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("FINAL AGGREGATED REPORT")
print("="*70)

print(result["final_report"])

# ============================================================
# END
# ============================================================

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



SIGNAL ANALYSIS AGENT
### 1. Problem Summary
The customer is experiencing extremely slow mobile internet and frequent call drops, which began occurring since yesterday evening. This indicates a potential issue with network performance and reliability in the area.

### 2. Root Cause
The symptoms described by the customer—slow internet and call drops—are likely caused by a combination of factors:

- **Tower Congestion**: Given that the issues started in the evening, it is possible that the network is experiencing high traffic during peak hours, leading to congestion. This can result in slow internet speeds and increased latency, which can contribute to call drops.
  
- **Regional Tower Outages**: There may be ongoing outages or maintenance affecting the local cell towers, which could impact multiple customers in the area. This would lead to degraded service quality for both mobile internet and voice calls.

- **Signal Quality**: If the customer is in an area with poor signal quality due

In [ ]:
# | Feature               | First System | Second System |
# | --------------------- | ------------ | ------------- |
# | Workflow Type         | Static       | Dynamic       |
# | Delegation            | Hardcoded    | AI-decided    |
# | Routing               | Fixed        | Conditional   |
# | Parallelism           | Yes          | Yes           |
# | Supervisor Agent      | ❌            | ✅             |
# | Team Selection        | Manual       | Autonomous    |
# | Conditional Execution | ❌            | ✅             |
# | Agent Orchestration   | Medium       | Advanced      |
# | Agentic Level         | Intermediate | Strong        |


In [ ]:
# ============================================================
# HIERARCHICAL MULTI-AGENT TELECOM COMMAND CENTER
# UPDATED FOR LATEST LANGCHAIN + LANGGRAPH
# ============================================================
#
# Architecture:
#
#                 Supervisor Agent
#                         ↓
#       ┌────────────────────────────────┐
#       ↓                ↓              ↓
#  Network Team    Billing Team   Security Team
#       ↓                ↓              ↓
#       └────────────────────────────────┘
#                         ↓
#                   Merge Agent
#
# ============================================================
#
# Features:
# - Latest LangChain Compatible
# - Latest LangGraph Compatible
# - Supervisor Delegation Pattern
# - Conditional Routing
# - GPT-4o Mini
# - text-embedding-3-small
# - Telecom RAG
# - Google Colab Optimized
# - OpenAI Key via Colab Secrets
#
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

# !pip install -q \
# langchain \
# langgraph \
# langchain-openai \
# langchain-community \
# langchain-core \
# langchain-text-splitters \
# faiss-cpu \
# tiktoken

# ============================================================
# 2. IMPORTS
# ============================================================

from typing import TypedDict, Dict, Any, List

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# ============================================================
# 3. LOAD OPENAI API KEY FROM GOOGLE COLAB SECRETS
# ============================================================

# Google Colab:
# Left Sidebar → Secrets
#
# Add:
# Name  : OPENAI_API_KEY
# Value : your-openai-api-key

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ============================================================
# 4. INITIALIZE GPT-4o MINI
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# ============================================================
# 5. INITIALIZE EMBEDDING MODEL
# ============================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# ============================================================
# 6. TELECOM KNOWLEDGE BASE
# ============================================================

telecom_docs = [

    """
    Tower congestion causes:
    - packet loss
    - high latency
    - call drops
    - slow internet
    during peak hours.
    """,

    """
    Telecom billing throttling occurs
    after Fair Usage Policy limits
    are exceeded.
    """,

    """
    SIM swap fraud allows attackers
    to hijack a customer's mobile number
    using unauthorized SIM replacement.
    """,

    """
    Telecom phishing and suspicious
    account access may indicate
    customer account compromise.
    """,

    """
    Expired telecom plans may affect:
    - voice services
    - mobile internet
    - enterprise connectivity
    """,

    """
    Regional tower outages impact:
    - multiple subscribers
    - enterprise clients
    - VoLTE services
    - internet stability
    """
]

# ============================================================
# 7. CREATE DOCUMENTS
# ============================================================

documents = [
    Document(page_content=doc)
    for doc in telecom_docs
]

# ============================================================
# 8. SPLIT DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)

# ============================================================
# 9. CREATE VECTOR DATABASE
# ============================================================

vectorstore = FAISS.from_documents(
    split_docs,
    embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# ============================================================
# 10. DEFINE SHARED GRAPH STATE
# ============================================================

class TelecomState(TypedDict):

    incident: str

    active_teams: List[str]

    network_report: Dict[str, Any]
    billing_report: Dict[str, Any]
    security_report: Dict[str, Any]

    final_summary: str

# ============================================================
# 11. HELPER FUNCTION FOR RETRIEVAL
# ============================================================

def retrieve_context(query: str):

    docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])

    return context

# ============================================================
# 12. SUPERVISOR AGENT
# ============================================================

def supervisor_agent(state: TelecomState):

    incident = state["incident"]

    prompt = f"""
    You are a Telecom Operations Supervisor AI.

    Incident:
    {incident}

    Available Teams:
    - network
    - billing
    - security

    Decide which teams should investigate.

    Return ONLY comma-separated team names.

    Example:
    network,security
    """

    response = llm.invoke(prompt)

    raw_output = response.content.strip().lower()

    active_teams = [
        team.strip()
        for team in raw_output.split(",")
    ]

    print("\n" + "="*60)
    print("SUPERVISOR DECISION")
    print("="*60)

    print(active_teams)

    return {
        "active_teams": active_teams
    }

# ============================================================
# 13. NETWORK TEAM AGENT
# ============================================================

def network_team(state: TelecomState):

    incident = state["incident"]

    context = retrieve_context(incident)

    prompt = f"""
    You are a Telecom Network Operations Expert.

    Incident:
    {incident}

    Telecom Context:
    {context}

    Analyze:
    - tower congestion
    - outages
    - packet loss
    - call drops
    - latency issues

    Generate:
    1. Network Issue
    2. Root Cause
    3. Severity
    """

    response = llm.invoke(prompt)

    return {
        "network_report": {
            "analysis": response.content
        }
    }

# ============================================================
# 14. BILLING TEAM AGENT
# ============================================================

def billing_team(state: TelecomState):

    incident = state["incident"]

    context = retrieve_context(incident)

    prompt = f"""
    You are a Telecom Billing Expert.

    Incident:
    {incident}

    Telecom Context:
    {context}

    Analyze:
    - billing issues
    - throttling
    - inactive plans
    - FUP violations

    Generate:
    1. Billing Diagnosis
    2. Possible Restrictions
    3. Recommended Actions
    """

    response = llm.invoke(prompt)

    return {
        "billing_report": {
            "analysis": response.content
        }
    }

# ============================================================
# 15. SECURITY TEAM AGENT
# ============================================================

def security_team(state: TelecomState):

    incident = state["incident"]

    context = retrieve_context(incident)

    prompt = f"""
    You are a Telecom Security Operations Expert.

    Incident:
    {incident}

    Telecom Context:
    {context}

    Analyze:
    - SIM swap fraud
    - suspicious account access
    - phishing indicators
    - account compromise

    Generate:
    1. Security Threat Analysis
    2. Risk Level
    3. Recommended Mitigation
    """

    response = llm.invoke(prompt)

    return {
        "security_report": {
            "analysis": response.content
        }
    }

# ============================================================
# 16. CONDITIONAL ROUTING FUNCTION
# ============================================================

def route_teams(state: TelecomState):

    teams = state["active_teams"]

    routes = []

    if "network" in teams:
        routes.append("network_team")

    if "billing" in teams:
        routes.append("billing_team")

    if "security" in teams:
        routes.append("security_team")

    return routes

# ============================================================
# 17. MERGE AGENT
# ============================================================

def merge_agent(state: TelecomState):

    network_report = state.get("network_report", {})
    billing_report = state.get("billing_report", {})
    security_report = state.get("security_report", {})

    prompt = f"""
    You are a Senior Telecom Incident Commander.

    Combine all department findings into
    one professional executive report.

    NETWORK REPORT:
    {network_report}

    BILLING REPORT:
    {billing_report}

    SECURITY REPORT:
    {security_report}

    Generate:
    1. Executive Summary
    2. Root Cause
    3. Severity
    4. Recommended Actions
    5. Confidence Score
    """

    response = llm.invoke(prompt)

    return {
        "final_summary": response.content
    }

# ============================================================
# 18. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(TelecomState)

# ============================================================
# 19. ADD NODES
# ============================================================

builder.add_node("supervisor_agent", supervisor_agent)

builder.add_node("network_team", network_team)

builder.add_node("billing_team", billing_team)

builder.add_node("security_team", security_team)

builder.add_node("merge_agent", merge_agent)

# ============================================================
# 20. DEFINE GRAPH FLOW
# ============================================================

# START → SUPERVISOR

builder.add_edge(START, "supervisor_agent")

# SUPERVISOR → CONDITIONAL TEAMS

builder.add_conditional_edges(
    "supervisor_agent",
    route_teams
)

# TEAM AGENTS → MERGE

builder.add_edge("network_team", "merge_agent")

builder.add_edge("billing_team", "merge_agent")

builder.add_edge("security_team", "merge_agent")

# MERGE → END

builder.add_edge("merge_agent", END)

# ============================================================
# 21. COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# 22. SAMPLE INCIDENT
# ============================================================

input_state = {

    "incident": """
    Multiple customers in Chennai are
    experiencing severe call drops and
    suspicious SIM replacement activity.

    Enterprise users also report unstable
    mobile internet connectivity.
    """
}

# ============================================================
# 23. RUN GRAPH
# ============================================================

result = graph.invoke(input_state)

# ============================================================
# 24. DISPLAY RESULTS
# ============================================================

print("\n" + "="*70)
print("ACTIVE TEAMS")
print("="*70)

print(result["active_teams"])

# ------------------------------------------------------------

if result.get("network_report"):

    print("\n" + "="*70)
    print("NETWORK TEAM REPORT")
    print("="*70)

    print(result["network_report"]["analysis"])

# ------------------------------------------------------------

if result.get("billing_report"):

    print("\n" + "="*70)
    print("BILLING TEAM REPORT")
    print("="*70)

    print(result["billing_report"]["analysis"])

# ------------------------------------------------------------

if result.get("security_report"):

    print("\n" + "="*70)
    print("SECURITY TEAM REPORT")
    print("="*70)

    print(result["security_report"]["analysis"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("FINAL EXECUTIVE INCIDENT REPORT")
print("="*70)

print(result["final_summary"])

# ============================================================
# END
# ============================================================


SUPERVISOR DECISION
['network', 'security']

ACTIVE TEAMS
['network', 'security']

NETWORK TEAM REPORT
### Network Issue
**Severe Call Drops and Unstable Mobile Internet Connectivity in Chennai**

### Root Cause
1. **Tower Congestion**: High traffic volume in the region, especially during peak hours, may lead to congestion at the base stations, resulting in call drops and unstable internet connectivity.
  
2. **Regional Tower Outages**: Outages affecting multiple towers in the area can lead to service disruptions for both individual and enterprise users. This can be due to maintenance, power failures, or equipment malfunctions.

3. **Packet Loss**: High levels of packet loss can occur due to network congestion or faulty equipment, leading to degraded voice quality and unstable internet connections.

4. **Expired Telecom Plans**: Customers with expired plans may experience restricted access to voice and data services, contributing to the perception of call drops and internet instabilit

In [ ]:
# ============================================================
# DYNAMIC ROUTING TELECOM SUPPORT SYSTEM
# USING LANGGRAPH + GPT-4o MINI
# ============================================================
#
# Architecture:
#
#               Customer Query
#                       ↓
#                 Router Agent
#                       ↓
#      ┌────────────────────────────────┐
#      ↓                ↓              ↓
# Network Agent   Billing Agent    SIM Agent
#      ↓                ↓              ↓
#      └────────────────────────────────┘
#                       ↓
#               Final Response Agent
#
# ============================================================
#
# Features:
# - Dynamic Routing
# - Conditional Edges
# - Intent Classification
# - Telecom RAG
# - GPT-4o Mini
# - text-embedding-3-small
# - Latest LangChain Compatible
# - Latest LangGraph Compatible
# - Google Colab Optimized
#
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

# !pip install -q \
# langchain \
# langgraph \
# langchain-openai \
# langchain-community \
# langchain-core \
# langchain-text-splitters \
# faiss-cpu \
# tiktoken

# ============================================================
# 2. IMPORTS
# ============================================================

from typing import TypedDict

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# ============================================================
# 3. LOAD OPENAI API KEY
# ============================================================

# Google Colab:
# Left Sidebar → Secrets
#
# Add:
# Name  : OPENAI_API_KEY
# Value : your-openai-api-key

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ============================================================
# 4. INITIALIZE GPT-4o MINI
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# ============================================================
# 5. INITIALIZE EMBEDDINGS
# ============================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# ============================================================
# 6. TELECOM KNOWLEDGE BASE
# ============================================================

telecom_docs = [

    """
    Network congestion causes:
    - slow internet
    - packet loss
    - high latency
    - call drops
    """,

    """
    Billing throttling occurs after
    Fair Usage Policy limits are exceeded.
    """,

    """
    SIM activation issues may occur due to:
    - failed authentication
    - damaged SIM
    - incorrect provisioning
    """,

    """
    Duplicate recharge charges may happen
    due to payment gateway delays or retries.
    """,

    """
    SIM replacement requests require
    customer verification and activation.
    """,

    """
    Tower outages affect:
    - internet speed
    - VoLTE
    - enterprise connectivity
    """
]

# ============================================================
# 7. CREATE DOCUMENTS
# ============================================================

documents = [
    Document(page_content=doc)
    for doc in telecom_docs
]

# ============================================================
# 8. SPLIT DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)

# ============================================================
# 9. CREATE VECTOR STORE
# ============================================================

vectorstore = FAISS.from_documents(
    split_docs,
    embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# ============================================================
# 10. DEFINE SHARED GRAPH STATE
# ============================================================

class TelecomState(TypedDict):

    customer_query: str

    selected_route: str

    specialist_response: str

    final_response: str

# ============================================================
# 11. HELPER FUNCTION FOR RAG
# ============================================================

def retrieve_context(query: str):

    docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])

    return context

# ============================================================
# 12. ROUTER AGENT
# ============================================================

def router_agent(state: TelecomState):

    query = state["customer_query"]

    prompt = f"""
    You are a Telecom Support Router.

    Customer Query:
    {query}

    Available Routes:
    - network_agent
    - billing_agent
    - sim_agent

    Routing Rules:

    network_agent:
    - slow internet
    - call drops
    - signal issues
    - tower outages

    billing_agent:
    - recharge problems
    - billing disputes
    - duplicate charges
    - throttling

    sim_agent:
    - SIM activation
    - SIM failure
    - SIM replacement
    - authentication problems

    Return ONLY the route name.
    """

    response = llm.invoke(prompt)

    route = response.content.strip()

    print("\n" + "="*60)
    print("ROUTER DECISION")
    print("="*60)

    print(route)

    return {
        "selected_route": route
    }

# ============================================================
# 13. CONDITIONAL ROUTING FUNCTION
# ============================================================

def route_query(state: TelecomState):

    return state["selected_route"]

# ============================================================
# 14. NETWORK SUPPORT AGENT
# ============================================================

def network_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom Network Support Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - tower congestion
    - network outages
    - latency
    - packet loss
    - call drops

    Generate:
    1. Problem Summary
    2. Root Cause
    3. Recommended Fix
    """

    response = llm.invoke(prompt)

    return {
        "specialist_response": response.content
    }

# ============================================================
# 15. BILLING SUPPORT AGENT
# ============================================================

def billing_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom Billing Support Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - duplicate charges
    - recharge failures
    - throttling
    - billing disputes

    Generate:
    1. Billing Diagnosis
    2. Cause
    3. Recommended Action
    """

    response = llm.invoke(prompt)

    return {
        "specialist_response": response.content
    }

# ============================================================
# 16. SIM SUPPORT AGENT
# ============================================================

def sim_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    prompt = f"""
    You are a Telecom SIM Support Expert.

    Customer Query:
    {query}

    Telecom Context:
    {context}

    Analyze:
    - SIM activation
    - authentication failure
    - SIM replacement
    - damaged SIM

    Generate:
    1. SIM Issue
    2. Root Cause
    3. Recommended Solution
    """

    response = llm.invoke(prompt)

    return {
        "specialist_response": response.content
    }

# ============================================================
# 17. FINAL RESPONSE AGENT
# ============================================================

def final_response_agent(state: TelecomState):

    route = state["selected_route"]

    specialist_response = state["specialist_response"]

    prompt = f"""
    You are a Senior Telecom Support Manager.

    Route Selected:
    {route}

    Specialist Report:
    {specialist_response}

    Generate:
    1. Executive Summary
    2. Final Recommendation
    3. Customer-Friendly Resolution

    Keep the tone professional and clear.
    """

    response = llm.invoke(prompt)

    return {
        "final_response": response.content
    }

# ============================================================
# 18. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(TelecomState)

# ============================================================
# 19. ADD NODES
# ============================================================

builder.add_node("router_agent", router_agent)

builder.add_node("network_agent", network_agent)

builder.add_node("billing_agent", billing_agent)

builder.add_node("sim_agent", sim_agent)

builder.add_node("final_response_agent", final_response_agent)

# ============================================================
# 20. DEFINE GRAPH FLOW
# ============================================================

# START → ROUTER

builder.add_edge(START, "router_agent")

# ROUTER → CONDITIONAL ROUTES

builder.add_conditional_edges(
    "router_agent",
    route_query
)

# SPECIALIST AGENTS → FINAL RESPONSE

builder.add_edge(
    "network_agent",
    "final_response_agent"
)

builder.add_edge(
    "billing_agent",
    "final_response_agent"
)

builder.add_edge(
    "sim_agent",
    "final_response_agent"
)

# FINAL RESPONSE → END

builder.add_edge(
    "final_response_agent",
    END
)

# ============================================================
# 21. COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# 22. SAMPLE CUSTOMER QUERY
# ============================================================

input_state = {

    "customer_query": """
    My mobile internet has become extremely slow
    and calls are dropping continuously.
    """
}

# ============================================================
# 23. RUN GRAPH
# ============================================================

result = graph.invoke(input_state)

# ============================================================
# 24. DISPLAY RESULTS
# ============================================================

print("\n" + "="*70)
print("SELECTED ROUTE")
print("="*70)

print(result["selected_route"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("SPECIALIST RESPONSE")
print("="*70)

print(result["specialist_response"])

# ------------------------------------------------------------

print("\n" + "="*70)
print("FINAL CUSTOMER RESPONSE")
print("="*70)

print(result["final_response"])

# ============================================================
# END
# ============================================================


ROUTER DECISION
network_agent

SELECTED ROUTE
network_agent

SPECIALIST RESPONSE
### 1. Problem Summary
The customer is experiencing extremely slow mobile internet speeds and frequent call drops. These issues are likely related to network congestion, potential tower outages, and associated factors such as high latency and packet loss.

### 2. Root Cause
- **Network Congestion**: High user traffic in the area may be overwhelming the available bandwidth, leading to slow internet speeds and increased latency. This congestion can also result in packet loss, which affects the quality of voice calls and can lead to call drops.
  
- **Tower Outages**: If there are any outages or maintenance activities affecting the nearby cell towers, this could significantly impact both internet connectivity and VoLTE (Voice over LTE) services, causing slow speeds and call drops.

- **High Latency**: Increased latency can occur due to congestion or routing issues, leading to delays in data transmission, whi

In [12]:
# | Capability             | System 1 | System 2 | System 3    |
# | ---------------------- | -------- | -------- | ----------- |
# | Parallel Agents        | ✅        | ✅        | ❌           |
# | Supervisor             | ❌        | ✅        | ❌           |
# | Dynamic Routing        | ❌        | ✅        | ✅           |
# | Multiple Specialists   | ✅        | ✅        | ❌           |
# | Collaboration          | Medium   | Strong   | Weak        |
# | Cost Efficient         | Medium   | Low      | High        |
# | Scalability            | Medium   | Medium   | High        |
# | Agentic Sophistication | Medium   | High     | Medium-High |


In [ ]:
# ============================================================
# RETRY & FALLBACK TELECOM AI SYSTEM
# USING LANGGRAPH + GPT-4o MINI
# ============================================================
#
# Architecture:
#
#               Customer Query
#                       ↓
#               Primary Agent
#                       ↓
#                 Success?
#                 ↓      ↓
#               Yes      No
#                ↓        ↓
#              END   Retry Controller
#                           ↓
#                    Retry Limit?
#                     ↓       ↓
#                   No       Yes
#                   ↓         ↓
#             Primary Agent  Fallback Agent
#                                   ↓
#                           Escalation Agent
#                                   ↓
#                                  END
#
# ============================================================
#
# Features:
# - Retry Logic
# - Fallback Workflow
# - Failure Recovery
# - Conditional Routing
# - Telecom RAG
# - GPT-4o Mini
# - text-embedding-3-small
# - Latest LangChain Compatible
# - Latest LangGraph Compatible
# - Google Colab Optimized
#
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q \
langchain \
langgraph \
langchain-openai \
langchain-community \
langchain-core \
langchain-text-splitters \
faiss-cpu \
tiktoken

# ============================================================
# 2. IMPORTS
# ============================================================

import random

from typing import TypedDict

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# ============================================================
# 3. LOAD OPENAI API KEY
# ============================================================

# Google Colab:
# Left Sidebar → Secrets
#
# Add:
# Name  : OPENAI_API_KEY
# Value : your-openai-api-key

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ============================================================
# 4. INITIALIZE GPT-4o MINI
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# ============================================================
# 5. INITIALIZE EMBEDDINGS
# ============================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# ============================================================
# 6. TELECOM KNOWLEDGE BASE
# ============================================================

telecom_docs = [

    """
    Network congestion can cause:
    - slow internet
    - packet loss
    - call drops
    - unstable connectivity
    """,

    """
    Tower outages affect:
    - LTE
    - VoLTE
    - enterprise connectivity
    - internet reliability
    """,

    """
    SIM authentication issues may
    impact mobile internet access.
    """,

    """
    Telecom recovery procedures include:
    - tower reset
    - route refresh
    - network balancing
    """,

    """
    Regional telecom failures may affect
    multiple enterprise users simultaneously.
    """
]

# ============================================================
# 7. CREATE DOCUMENTS
# ============================================================

documents = [
    Document(page_content=doc)
    for doc in telecom_docs
]

# ============================================================
# 8. SPLIT DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)

# ============================================================
# 9. CREATE VECTOR STORE
# ============================================================

vectorstore = FAISS.from_documents(
    split_docs,
    embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# ============================================================
# 10. DEFINE SHARED GRAPH STATE
# ============================================================

class TelecomState(TypedDict):

    customer_query: str

    retry_count: int

    status: str

    primary_response: str

    fallback_response: str

    final_response: str

# ============================================================
# 11. HELPER FUNCTION FOR RAG
# ============================================================

def retrieve_context(query: str):

    docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])

    return context

# ============================================================
# 12. PRIMARY TELECOM AGENT
# ============================================================

def primary_agent(state: TelecomState):

    query = state["customer_query"]

    retry_count = state["retry_count"]

    print("\n" + "="*60)
    print(f"PRIMARY AGENT ATTEMPT: {retry_count + 1}")
    print("="*60)

    # --------------------------------------------------------
    # SIMULATED FAILURE LOGIC
    # --------------------------------------------------------

    simulated_failure = random.choice([True, False])

    if simulated_failure:

        print("Primary Agent Failed!")

        return {
            "status": "failure"
        }

    # --------------------------------------------------------
    # SUCCESS PATH
    # --------------------------------------------------------

    prompt = f"""
    You are a Telecom Diagnostics Expert.

    Customer Query:
    {query}

    Analyze:
    - network issue
    - connectivity problem
    - possible outage

    Generate:
    1. Problem Summary
    2. Root Cause
    3. Recommended Action
    """

    response = llm.invoke(prompt)

    print("Primary Agent Succeeded!")

    return {
        "status": "success",
        "primary_response": response.content
    }

# ============================================================
# 13. ROUTING AFTER PRIMARY AGENT
# ============================================================

def check_primary_status(state: TelecomState):

    status = state["status"]

    if status == "success":
        return "final_response_agent"

    return "retry_controller"

# ============================================================
# 14. RETRY CONTROLLER
# ============================================================

MAX_RETRIES = 2

def retry_controller(state: TelecomState):

    retry_count = state["retry_count"]

    retry_count += 1

    print("\nRetry Count:", retry_count)

    return {
        "retry_count": retry_count
    }

# ============================================================
# 15. RETRY ROUTING LOGIC
# ============================================================

def retry_router(state: TelecomState):

    retry_count = state["retry_count"]

    if retry_count <= MAX_RETRIES:
        return "primary_agent"

    return "fallback_agent"

# ============================================================
# 16. FALLBACK AGENT
# ============================================================

def fallback_agent(state: TelecomState):

    query = state["customer_query"]

    context = retrieve_context(query)

    print("\nFallback Agent Activated!")

    prompt = f"""
    You are a Telecom Recovery Specialist.

    Customer Query:
    {query}

    Telecom Knowledge Base:
    {context}

    Since the primary system failed,
    use the knowledge base to provide
    fallback troubleshooting guidance.

    Generate:
    1. Possible Issue
    2. Recovery Suggestion
    3. Recommended Escalation
    """

    response = llm.invoke(prompt)

    return {
        "fallback_response": response.content
    }

# ============================================================
# 17. ESCALATION AGENT
# ============================================================

def escalation_agent(state: TelecomState):

    fallback_response = state["fallback_response"]

    prompt = f"""
    You are a Telecom Incident Escalation Manager.

    Fallback Report:
    {fallback_response}

    Generate:
    1. Escalation Decision
    2. Severity Level
    3. Human Support Recommendation
    """

    response = llm.invoke(prompt)

    return {
        "final_response": response.content
    }

# ============================================================
# 18. FINAL RESPONSE AGENT
# ============================================================

def final_response_agent(state: TelecomState):

    primary_response = state["primary_response"]

    prompt = f"""
    You are a Senior Telecom Operations Manager.

    Diagnostics Report:
    {primary_response}

    Generate:
    1. Executive Summary
    2. Final Resolution
    3. Customer-Friendly Explanation
    """

    response = llm.invoke(prompt)

    return {
        "final_response": response.content
    }

# ============================================================
# 19. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(TelecomState)

# ============================================================
# 20. ADD NODES
# ============================================================

builder.add_node(
    "primary_agent",
    primary_agent
)

builder.add_node(
    "retry_controller",
    retry_controller
)

builder.add_node(
    "fallback_agent",
    fallback_agent
)

builder.add_node(
    "escalation_agent",
    escalation_agent
)

builder.add_node(
    "final_response_agent",
    final_response_agent
)

# ============================================================
# 21. DEFINE GRAPH FLOW
# ============================================================

# START → PRIMARY AGENT

builder.add_edge(
    START,
    "primary_agent"
)

# PRIMARY → CONDITIONAL CHECK

builder.add_conditional_edges(
    "primary_agent",
    check_primary_status
)

# RETRY CONTROLLER ROUTING

builder.add_conditional_edges(
    "retry_controller",
    retry_router
)

# FALLBACK → ESCALATION

builder.add_edge(
    "fallback_agent",
    "escalation_agent"
)

# ESCALATION → END

builder.add_edge(
    "escalation_agent",
    END
)

# FINAL RESPONSE → END

builder.add_edge(
    "final_response_agent",
    END
)

# ============================================================
# 22. COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# 23. SAMPLE CUSTOMER QUERY
# ============================================================

input_state = {

    "customer_query": """
    My mobile network suddenly stopped working
    and internet connectivity is unstable.
    """,

    "retry_count": 0,

    "status": "",

    "primary_response": "",

    "fallback_response": "",

    "final_response": ""
}

# ============================================================
# 24. RUN GRAPH
# ============================================================

result = graph.invoke(input_state)

# ============================================================
# 25. DISPLAY RESULTS
# ============================================================

print("\n" + "="*70)
print("FINAL SYSTEM RESPONSE")
print("="*70)

print(result["final_response"])

# ============================================================
# END
# ============================================================


PRIMARY AGENT ATTEMPT: 1
Primary Agent Succeeded!

FINAL SYSTEM RESPONSE
### 1. Executive Summary
The customer has reported a sudden loss of mobile network functionality, which includes disruptions in voice calls, text messaging, and internet connectivity. A thorough diagnostics report indicates that the root cause may be attributed to network issues, device connectivity problems, or potential outages in the area. To address this, a series of recommended actions have been outlined, including checking for outages, restarting the device, verifying network settings, reinserting the SIM card, updating software, and contacting customer support if the issue persists. The goal is to restore full communication capabilities for the customer as swiftly as possible.

### 2. Final Resolution
After following the recommended troubleshooting steps, the customer successfully restored their mobile network functionality. The issue was identified as a temporary network congestion that resolved itself af

In [13]:
# | Feature / Capability       | System 1 — Parallel Multi-Agent    | System 2 — Hierarchical Supervisor | System 3 — Dynamic Router   | System 4 — Retry & Fallback  |
# | -------------------------- | ---------------------------------- | ---------------------------------- | --------------------------- | ---------------------------- |
# | Core Pattern               | Parallel execution                 | Supervisor delegation              | Intent routing              | Fault-tolerant workflow      |
# | Main Goal                  | Analyze from multiple perspectives | Dynamically coordinate teams       | Route to correct specialist | Recover from failures        |
# | Architecture Style         | Static graph                       | Hierarchical graph                 | Conditional routing graph   | Adaptive recovery graph      |
# | Entry Point                | START → All agents                 | START → Supervisor                 | START → Router              | START → Primary agent        |
# | Decision Maker             | None                               | Supervisor agent                   | Router agent                | Retry controller             |
# | Dynamic Routing            | ❌                                  | ✅                                  | ✅                           | ✅                            |
# | Conditional Edges          | ❌                                  | ✅                                  | ✅                           | ✅                            |
# | Parallel Execution         | ✅                                  | ✅                                  | ❌                           | ❌                            |
# | Multiple Agents at Once    | ✅                                  | ✅                                  | ❌                           | Partial                      |
# | AI Delegation              | ❌                                  | ✅                                  | Partial                     | Partial                      |
# | Agent Collaboration        | Medium                             | High                               | Low                         | Low                          |
# | Shared State Memory        | ✅                                  | ✅                                  | ✅                           | ✅                            |
# | Runtime Adaptation         | ❌                                  | ✅                                  | ✅                           | ✅                            |
# | Retry Logic                | ❌                                  | ❌                                  | ❌                           | ✅                            |
# | Fallback Strategy          | ❌                                  | ❌                                  | ❌                           | ✅                            |
# | Escalation Workflow        | ❌                                  | ❌                                  | ❌                           | ✅                            |
# | Failure Recovery           | ❌                                  | ❌                                  | ❌                           | ✅                            |
# | Operational Resilience     | Low                                | Medium                             | Medium                      | High                         |
# | Cost Efficiency            | Medium                             | Low                                | High                        | Medium                       |
# | Complexity                 | Medium                             | High                               | Medium                      | High                         |
# | Scalability                | Medium                             | Medium                             | High                        | High                         |
# | Best Use Case              | Deep multi-analysis                | Enterprise coordination            | Customer support routing    | Production reliability       |
# | Real-World Analogy         | Multiple experts working together  | Manager coordinating departments   | Receptionist routing calls  | SRE incident recovery system |
# | Closest Enterprise Pattern | Parallel diagnostics               | AI command center                  | AI triage/helpdesk          | AI reliability orchestration |
# | Agentic Sophistication     | Medium                             | High                               | Medium-High                 | High                         |
# | Autonomous Decision Making | Low                                | High                               | Medium                      | Medium-High                  |
# | Looping / Feedback Cycles  | ❌                                  | ❌                                  | ❌                           | ✅                            |
# | Workflow Type              | Static                             | Dynamic                            | Dynamic                     | Adaptive                     |
# | Collaboration Depth        | Medium                             | Strong                             | Weak                        | Weak                         |
# | Production Readiness       | Medium                             | High                               | Very High                   | Very High                    |
# | Best Characterization      | Multi-Agent Workflow               | Hierarchical Multi-Agent System    | Dynamic Routing Agent       | Fault-Tolerant Agent System  |
# | “True AI Agent”?           | ✅ Yes                              | ✅ Strong Yes                       | ✅ Yes                       | ✅ Strong Yes                 |


In [2]:
# ============================================================
# HUMAN-IN-THE-LOOP (HITL) TELECOM FRAUD SYSTEM
# USING LANGGRAPH + GPT-4o MINI
# ============================================================
#
# Architecture:
#
#              Customer Activity
#                      ↓
#           Fraud Detection Agent
#                      ↓
#             Risk Analysis Agent
#                      ↓
#              Human Approval Check
#                 ↓            ↓
#              Low Risk     High Risk
#                 ↓             ↓
#            Auto Approve   Human Review
#                                 ↓
#                      Approve / Reject
#                                 ↓
#                          Final Decision
#
# ============================================================
#
# Features:
# - Human-in-the-Loop Workflow
# - Fraud Detection
# - Risk Scoring
# - Conditional Routing
# - Telecom Fraud RAG
# - GPT-4o Mini
# - text-embedding-3-small
# - Latest LangChain Compatible
# - Latest LangGraph Compatible
# - Google Colab Optimized
#
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

# !pip install -q \
# langchain \
# langgraph \
# langchain-openai \
# langchain-community \
# langchain-core \
# langchain-text-splitters \
# faiss-cpu \
# tiktoken

# ============================================================
# 2. IMPORTS
# ============================================================

import random

from typing import TypedDict

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# ============================================================
# 3. LOAD OPENAI API KEY
# ============================================================

# Google Colab:
# Left Sidebar → Secrets
#
# Add:
# Name  : OPENAI_API_KEY
# Value : your-openai-api-key

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ============================================================
# 4. INITIALIZE GPT-4o MINI
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# ============================================================
# 5. INITIALIZE EMBEDDINGS
# ============================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# ============================================================
# 6. TELECOM FRAUD KNOWLEDGE BASE
# ============================================================

telecom_docs = [

    """
    SIM swap fraud occurs when attackers
    gain control of a user's mobile number
    using unauthorized SIM replacement.
    """,

    """
    Sudden device change combined with
    location change may indicate
    telecom account compromise.
    """,

    """
    Multiple OTP requests in a short time
    can indicate suspicious activity.
    """,

    """
    Fraud prevention procedures include:
    - temporary account freeze
    - identity verification
    - escalation to fraud analysts
    """,

    """
    Telecom fraud investigations require:
    - audit logs
    - customer verification
    - manual review
    """
]

# ============================================================
# 7. CREATE DOCUMENTS
# ============================================================

documents = [
    Document(page_content=doc)
    for doc in telecom_docs
]

# ============================================================
# 8. SPLIT DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)

# ============================================================
# 9. CREATE VECTOR STORE
# ============================================================

vectorstore = FAISS.from_documents(
    split_docs,
    embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# ============================================================
# 10. DEFINE SHARED GRAPH STATE
# ============================================================

class TelecomState(TypedDict):

    customer_activity: str

    fraud_analysis: str

    fraud_score: int

    risk_level: str

    requires_human: bool

    human_decision: str

    final_action: str

# ============================================================
# 11. HELPER FUNCTION FOR RAG
# ============================================================

def retrieve_context(query: str):

    docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])

    return context

# ============================================================
# 12. FRAUD DETECTION AGENT
# ============================================================

def fraud_detection_agent(state: TelecomState):

    activity = state["customer_activity"]

    context = retrieve_context(activity)

    prompt = f"""
    You are a Telecom Fraud Detection Expert.

    Customer Activity:
    {activity}

    Telecom Fraud Context:
    {context}

    Analyze:
    - suspicious SIM activity
    - unusual location changes
    - device changes
    - OTP abuse indicators

    Generate:
    1. Fraud Indicators
    2. Threat Summary
    3. Initial Assessment
    """

    response = llm.invoke(prompt)

    return {
        "fraud_analysis": response.content
    }

# ============================================================
# 13. RISK ANALYSIS AGENT
# ============================================================

def risk_analysis_agent(state: TelecomState):

    fraud_analysis = state["fraud_analysis"]

    prompt = f"""
    You are a Telecom Fraud Risk Analyst.

    Fraud Analysis:
    {fraud_analysis}

    Generate:
    1. Fraud Risk Score (0-100)
    2. Risk Level
    3. Whether Human Approval is Required

    Return in EXACT format:

    SCORE: <number>
    RISK: <LOW/MEDIUM/HIGH>
    HUMAN_REQUIRED: <YES/NO>
    """

    response = llm.invoke(prompt)

    content = response.content

    # --------------------------------------------------------
    # SIMPLE PARSING
    # --------------------------------------------------------

    score = 50
    risk = "MEDIUM"
    requires_human = False

    lines = content.split("\n")

    for line in lines:

        if "SCORE:" in line:
            try:
                score = int(
                    line.split(":")[1].strip()
                )
            except:
                pass

        if "RISK:" in line:
            risk = line.split(":")[1].strip()

        if "HUMAN_REQUIRED:" in line:

            value = line.split(":")[1].strip()

            if value == "YES":
                requires_human = True

    print("\n" + "="*60)
    print("RISK ANALYSIS")
    print("="*60)

    print("Fraud Score:", score)
    print("Risk Level:", risk)
    print("Human Approval Required:", requires_human)

    return {
        "fraud_score": score,
        "risk_level": risk,
        "requires_human": requires_human
    }

# ============================================================
# 14. HUMAN APPROVAL ROUTER
# ============================================================

def approval_router(state: TelecomState):

    if state["requires_human"]:
        return "human_approval_node"

    return "final_decision_agent"

# ============================================================
# 15. HUMAN APPROVAL NODE
# ============================================================

def human_approval_node(state: TelecomState):

    print("\n" + "="*60)
    print("HUMAN REVIEW REQUIRED")
    print("="*60)

    print("\nFraud Score:", state["fraud_score"])
    print("Risk Level:", state["risk_level"])

    print("\nCustomer Activity:")
    print(state["customer_activity"])

    # --------------------------------------------------------
    # HUMAN INPUT
    # --------------------------------------------------------

    decision = input(
        "\nEnter Decision (approve/reject): "
    ).strip().lower()

    return {
        "human_decision": decision
    }

# ============================================================
# 16. FINAL DECISION AGENT
# ============================================================

def final_decision_agent(state: TelecomState):

    fraud_score = state["fraud_score"]

    risk_level = state["risk_level"]

    human_decision = state.get(
        "human_decision",
        "auto-approved"
    )

    activity = state["customer_activity"]

    prompt = f"""
    You are a Senior Telecom Fraud Manager.

    Customer Activity:
    {activity}

    Fraud Score:
    {fraud_score}

    Risk Level:
    {risk_level}

    Human Decision:
    {human_decision}

    Generate:
    1. Final Fraud Assessment
    2. Account Action
    3. Customer Notification
    4. Escalation Recommendation
    """

    response = llm.invoke(prompt)

    return {
        "final_action": response.content
    }

# ============================================================
# 17. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(TelecomState)

# ============================================================
# 18. ADD NODES
# ============================================================

builder.add_node(
    "fraud_detection_agent",
    fraud_detection_agent
)

builder.add_node(
    "risk_analysis_agent",
    risk_analysis_agent
)

builder.add_node(
    "human_approval_node",
    human_approval_node
)

builder.add_node(
    "final_decision_agent",
    final_decision_agent
)

# ============================================================
# 19. DEFINE GRAPH FLOW
# ============================================================

# START → FRAUD DETECTION

builder.add_edge(
    START,
    "fraud_detection_agent"
)

# FRAUD DETECTION → RISK ANALYSIS

builder.add_edge(
    "fraud_detection_agent",
    "risk_analysis_agent"
)

# RISK ANALYSIS → CONDITIONAL APPROVAL

builder.add_conditional_edges(
    "risk_analysis_agent",
    approval_router
)

# HUMAN APPROVAL → FINAL DECISION

builder.add_edge(
    "human_approval_node",
    "final_decision_agent"
)

# FINAL DECISION → END

builder.add_edge(
    "final_decision_agent",
    END
)

# ============================================================
# 20. COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# 21. SAMPLE CUSTOMER ACTIVITY
# ============================================================

input_state = {

    "customer_activity": """
    Customer changed SIM card, switched device,
    changed location from Chennai to Delhi,
    and requested multiple OTP resets
    within 15 minutes.
    """,

    "fraud_analysis": "",

    "fraud_score": 0,

    "risk_level": "",

    "requires_human": False,

    "human_decision": "",

    "final_action": ""
}

# ============================================================
# 22. RUN GRAPH
# ============================================================

result = graph.invoke(input_state)

# ============================================================
# 23. DISPLAY RESULTS
# ============================================================

print("\n" + "="*70)
print("FINAL FRAUD SYSTEM DECISION")
print("="*70)

print(result["final_action"])

# ============================================================
# END
# ============================================================


RISK ANALYSIS
Fraud Score: 85
Risk Level: HIGH
Human Approval Required: True

HUMAN REVIEW REQUIRED

Fraud Score: 85
Risk Level: HIGH

Customer Activity:

    Customer changed SIM card, switched device,
    changed location from Chennai to Delhi,
    and requested multiple OTP resets
    within 15 minutes.
    

Enter Decision (approve/reject): approve

FINAL FRAUD SYSTEM DECISION
### 1. Final Fraud Assessment
Based on the customer activity observed, the following factors contribute to the high fraud score of 85 and the classification of the risk level as HIGH:

- **SIM Card Change**: Changing the SIM card can indicate potential account takeover or unauthorized access.
- **Device Switch**: Switching devices frequently can be a red flag, especially if it occurs in conjunction with other suspicious activities.
- **Location Change**: The sudden change in location from Chennai to Delhi raises concerns about the legitimacy of the account usage, particularly if the customer has not previous

In [15]:
# | # | Architecture                   | Core Pattern                      | Workflow Style                | Agentic Level | Key Strength                   | Main Decision Maker   | Parallelism | Dynamic Routing | Retry/Fallback | Human Oversight | Collaboration Level | Enterprise Realism | Best Use Case                 | Complexity |
# | - | ------------------------------ | --------------------------------- | ----------------------------- | ------------- | ------------------------------ | --------------------- | ----------- | --------------- | -------------- | --------------- | ------------------- | ------------------ | ----------------------------- | ---------- |
# | 1 | Parallel Multi-Agent System    | Multiple specialists run together | Static Workflow               | Medium        | Multi-perspective analysis     | None                  | ✅ Yes       | ❌ No            | ❌ No           | ❌ No            | Medium              | Medium             | Diagnostics & analysis        | Medium     |
# | 2 | Hierarchical Supervisor System | Supervisor delegates teams        | Dynamic Hierarchical Workflow | Very High     | Delegation & orchestration     | Supervisor Agent      | ✅ Yes       | ✅ Yes           | ❌ No           | ❌ No            | Very High           | Very High          | Enterprise command centers    | High       |
# | 3 | Dynamic Routing System         | Intent-based routing              | Conditional Workflow          | Medium-High   | Efficient specialist selection | Router Agent          | ❌ No        | ✅ Yes           | ❌ No           | ❌ No            | Low                 | Very High          | Customer support routing      | Medium     |
# | 4 | Retry & Fallback System        | Adaptive recovery workflow        | Fault-Tolerant Workflow       | High          | Reliability & resilience       | Retry Controller      | ❌ No        | ✅ Yes           | ✅ Yes          | ❌ No            | Low                 | Very High          | Production AI reliability     | High       |
# | 5 | Human-in-the-Loop System       | AI + human governance             | Governed Approval Workflow    | Very High     | Safety & oversight             | Risk + Human Reviewer | ❌ No        | ✅ Yes           | Partial        | ✅ Yes           | Medium              | Extremely High     | Fraud/risk/compliance systems | High       |


In [16]:
# Enterprise AI Maturity Mapping

# | Capability             | S1     | S2      | S3        | S4        | S5             |
# | ---------------------- | ------ | ------- | --------- | --------- | -------------- |
# | Multi-Agent Execution  | ✅      | ✅       | Partial   | Partial   | ✅              |
# | Supervisor Delegation  | ❌      | ✅       | ❌         | ❌         | Partial        |
# | Dynamic Routing        | ❌      | ✅       | ✅         | ✅         | ✅              |
# | Parallel Collaboration | ✅      | ✅       | ❌         | ❌         | ❌              |
# | Shared State Memory    | ✅      | ✅       | ✅         | ✅         | ✅              |
# | Conditional Logic      | ❌      | ✅       | ✅         | ✅         | ✅              |
# | Retry Loops            | ❌      | ❌       | ❌         | ✅         | ❌              |
# | Fallback Handling      | ❌      | ❌       | ❌         | ✅         | Partial        |
# | Escalation Workflow    | ❌      | ❌       | ❌         | ✅         | ✅              |
# | Risk-Based Decisions   | ❌      | Partial | ❌         | Partial   | ✅              |
# | Human Oversight        | ❌      | ❌       | ❌         | ❌         | ✅              |
# | Governance Controls    | ❌      | ❌       | ❌         | Partial   | ✅              |
# | Production Readiness   | Medium | High    | Very High | Very High | Extremely High |
